# SolarMap — train the panel segmentation model

Trains the U-Net that `scripts/detect.py` and the web UI need, and hands back a
`solar_unet.pt` checkpoint to drop into your local `models/` folder.

**Run this on Colab with a GPU** — Runtime ▸ Change runtime type ▸ **T4 GPU**.
On CPU this takes many hours; on a T4 it is roughly 40–60 minutes.

Dataset: [BDAPPV](https://zenodo.org/records/7358126) (Kasmi et al. 2023,
*Scientific Data*, CC-BY-4.0) — ~28k aerial images with hand-drawn PV masks.
The `google` split is imagery from Google, which is the closest match to what
the tile backends capture.

## 1 · Confirm the GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    print("\nNO GPU. Runtime > Change runtime type > T4 GPU, then rerun.")

## 2 · Dependencies

In [ ]:
!pip -q install segmentation-models-pytorch albumentations
import segmentation_models_pytorch as smp, albumentations as A
print("smp", smp.__version__, "| albumentations", A.__version__)

## 3 · Download BDAPPV

`bdappv.zip` is ~8 GB, so this is the slow step (typically 10–20 min on Colab).
The download resumes if it is interrupted — just rerun the cell.

In [ ]:
import os, requests
from pathlib import Path

REC = "7358126"
DEST = Path("/content/bdappv.zip")

files = {f["key"]: f for f in requests.get(f"https://zenodo.org/api/records/{REC}").json()["files"]}
target = files["bdappv.zip"]
url, size = target["links"]["self"], target["size"]
print(f"{size/1e9:.1f} GB from {url}")

done = DEST.stat().st_size if DEST.exists() else 0
if done < size:
    # Range request so an interrupted download picks up where it stopped.
    headers = {"Range": f"bytes={done}-"} if done else {}
    with requests.get(url, headers=headers, stream=True) as r, open(DEST, "ab") as fh:
        for chunk in r.iter_content(1 << 22):
            fh.write(chunk)
            done += len(chunk)
            print(f"\r{done/1e9:.2f}/{size/1e9:.2f} GB", end="")
print(f"\ndownloaded: {DEST.stat().st_size/1e9:.2f} GB")

In [ ]:
import zipfile
from pathlib import Path

ROOT = Path("/content/bdappv")
if not ROOT.exists():
    with zipfile.ZipFile("/content/bdappv.zip") as z:
        z.extractall("/content/")
print("extracted. top level:", sorted(p.name for p in Path("/content").glob("bdappv*")))
for p in sorted(ROOT.rglob("*"))[:12]:
    if p.is_dir():
        n = len(list(p.glob("*")))
        print(f"  {p.relative_to(ROOT)}/  ({n} entries)")

## 4 · Build the train/val split

BDAPPV ships many **unannotated** images alongside the labelled ones. Those are
kept as all-zero masks, capped at the number of positives — they are what
teaches the model that swimming pools, skylights and dark flat roofs are *not*
panels. Without them you get a detector that flags every dark rectangle.

In [ ]:
import random, shutil
import numpy as np
from PIL import Image
from pathlib import Path

SRC = Path("/content/bdappv/google")     # adjust if the layout above differs
OUT = Path("/content/dataset")
VAL_FRAC, SEED = 0.15, 1234

img_dir = next(SRC/d for d in ("img","images") if (SRC/d).is_dir())
mask_dir = next(SRC/d for d in ("mask","masks") if (SRC/d).is_dir())
masks = {p.stem: p for p in mask_dir.iterdir() if p.is_file()}
images = sorted(p for p in img_dir.iterdir() if p.is_file())

pos = [p for p in images if p.stem in masks]
neg = [p for p in images if p.stem not in masks]
random.Random(SEED).shuffle(neg)
neg = neg[:len(pos)]
print(f"{len(pos)} annotated, {len(neg)} negatives kept")

items = [(p, masks[p.stem]) for p in pos] + [(p, None) for p in neg]
random.Random(SEED).shuffle(items)
split = int(len(items) * (1 - VAL_FRAC))

for s in ("train/images","train/masks","val/images","val/masks"):
    (OUT/s).mkdir(parents=True, exist_ok=True)

for i, (ip, mp) in enumerate(items):
    sp = "train" if i < split else "val"
    shutil.copyfile(ip, OUT/sp/"images"/f"{ip.stem}.png") if ip.suffix==".png" \
        else Image.open(ip).convert("RGB").save(OUT/sp/"images"/f"{ip.stem}.png")
    if mp is None:
        w,h = Image.open(ip).size
        Image.fromarray(np.zeros((h,w), np.uint8)).save(OUT/sp/"masks"/f"{ip.stem}.png")
    else:
        a = np.array(Image.open(mp).convert("L"))
        Image.fromarray(((a>0)*255).astype(np.uint8)).save(OUT/sp/"masks"/f"{ip.stem}.png")

print(f"train={split}  val={len(items)-split}")

## 5 · Model, data pipeline and loss

Kept deliberately identical to `src/solarmap/model/` so the checkpoint this
produces loads without surprises. **If you change the encoder here, change it
in `config.yaml` too.**

In [ ]:
import cv2, numpy as np, torch, torch.nn as nn
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

ENCODER, TILE, BATCH, EPOCHS, LR = "resnet34", 512, 16, 30, 3e-4

params = smp.encoders.get_preprocessing_params(ENCODER, "imagenet")
MEAN, STD = tuple(params["mean"]), tuple(params["std"])

def build_tf(train):
    stages = [
        A.RandomResizedCrop(size=(TILE,TILE), scale=(0.7,1.0), ratio=(0.9,1.11)),
        # Overhead imagery has no canonical orientation, so the full dihedral
        # group is valid augmentation rather than a distortion.
        A.HorizontalFlip(p=.5), A.VerticalFlip(p=.5), A.RandomRotate90(p=1.),
        # Mosaics stitch captures from different dates, sensors and sun angles;
        # this is the dominant domain shift between BDAPPV and your imagery.
        A.RandomBrightnessContrast(.25,.25,p=.7), A.HueSaturationValue(10,20,12,p=.4),
        A.GaussNoise(p=.2), A.MotionBlur(blur_limit=3, p=.15),
    ] if train else [A.Resize(TILE,TILE)]
    return A.Compose(stages + [A.Normalize(mean=MEAN, std=STD), ToTensorV2()])

class SegDS(Dataset):
    def __init__(self, root, split):
        self.imgs = sorted((Path(root)/split/"images").glob("*"))
        self.mdir = Path(root)/split/"masks"
        self.tf = build_tf(split=="train")
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        p = self.imgs[i]
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        m = (cv2.imread(str(self.mdir/p.name), cv2.IMREAD_GRAYSCALE) > 127).astype(np.float32)
        o = self.tf(image=img, mask=m)
        return o["image"], o["mask"].unsqueeze(0)

class DiceBCE(nn.Module):
    # Panels cover a small fraction of any tile. BCE alone scores well by
    # predicting "roof everywhere"; Dice makes it pay for missing positives.
    def __init__(self, w=.5): super().__init__(); self.w=w; self.bce=nn.BCEWithLogitsLoss()
    def forward(self, logits, t):
        p = torch.sigmoid(logits)
        num = 2*(p*t).sum((1,2,3)) + 1
        den = p.sum((1,2,3)) + t.sum((1,2,3)) + 1
        return self.w*self.bce(logits,t) + (1-self.w)*(1-(num/den).mean())

print("mean", MEAN, "std", STD)

## 6 · Train

In [ ]:
from pathlib import Path
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tr = DataLoader(SegDS("/content/dataset","train"), BATCH, shuffle=True, num_workers=2, drop_last=True)
va = DataLoader(SegDS("/content/dataset","val"), BATCH, num_workers=2)
print(f"train={len(tr.dataset)} val={len(va.dataset)} device={dev}")

model = smp.Unet(ENCODER, encoder_weights="imagenet", in_channels=3, classes=1).to(dev)
crit, opt = DiceBCE(.5), torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.amp.GradScaler(dev.type, enabled=dev.type=="cuda")

best = -1.0
for ep in range(1, EPOCHS+1):
    model.train(); run = 0.
    for x,y in tr:
        x,y = x.to(dev), y.to(dev)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(dev.type, enabled=dev.type=="cuda"):
            loss = crit(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); run += loss.item()
    sched.step()

    model.eval(); inter=union=0.
    with torch.no_grad():
        for x,y in va:
            x,y = x.to(dev), y.to(dev)
            p = (torch.sigmoid(model(x)) > .5).float()
            inter += (p*y).sum().item(); union += (p.sum()+y.sum()-(p*y).sum()).item()
    iou = inter/union if union else 1.
    print(f"epoch {ep:3d}  loss={run/len(tr):.4f}  val_IoU={iou:.4f}")

    if iou > best:
        best = iou
        # Field names must match what src/solarmap/infer/predict.py reads.
        torch.save({"state_dict": model.state_dict(), "encoder": ENCODER,
                    "tile_size": TILE, "mean": MEAN, "std": STD,
                    "val_iou": iou, "epoch": ep}, "/content/solar_unet.pt")
        print(f"   saved (val_IoU={iou:.4f})")
print(f"\nbest val_IoU = {best:.4f}")

## 7 · Per-array precision and recall

Validation IoU measures **pixels**. It does not tell you *"of 100 real arrays,
how many did we find, and how many pools did we call solar"* — and those are
the numbers that decide whether the output is trustworthy. This cell matches
predicted blobs to ground-truth blobs and sweeps the threshold so you can pick
an operating point deliberately instead of defaulting to 0.5.

In [ ]:
import cv2, numpy as np, torch

def blobs(mask):
    n, lab = cv2.connectedComponents(mask.astype(np.uint8))
    return [(lab==i) for i in range(1, n)]

def match(pred, gt, iou_min=.3):
    P, G = blobs(pred), blobs(gt)
    used, tp = set(), 0
    for p in P:
        for j, g in enumerate(G):
            if j in used: continue
            inter = (p&g).sum()
            if inter and inter/((p|g).sum()) >= iou_min:
                used.add(j); tp += 1; break
    return tp, len(P)-tp, len(G)-len(used)      # tp, fp, fn

model.eval()
print(f"{'thr':>5} {'precision':>10} {'recall':>8} {'F1':>7}")
for thr in (0.3, 0.4, 0.5, 0.6, 0.7):
    TP=FP=FN=0
    with torch.no_grad():
        for i,(x,y) in enumerate(va):
            if i >= 12: break                    # ~12 batches is enough to rank thresholds
            pr = (torch.sigmoid(model(x.to(dev))) > thr).cpu().numpy()[:,0]
            gt = y.numpy()[:,0]
            for a,b in zip(pr, gt):
                t,f,n = match(a, b>.5); TP+=t; FP+=f; FN+=n
    prec = TP/(TP+FP) if TP+FP else 0
    rec  = TP/(TP+FN) if TP+FN else 0
    f1   = 2*prec*rec/(prec+rec) if prec+rec else 0
    print(f"{thr:5.2f} {prec:10.3f} {rec:8.3f} {f1:7.3f}")
print("\nPut your chosen threshold in config.yaml under inference.threshold.")

## 7b · Fine-tune on your Lebanese labels

Training on BDAPPV alone gives a model that understands rooftop PV in general,
from ~28,000 European examples. It will still be slightly off for Lebanon --
different roof materials, sun angle and imagery source.

Fine-tuning on the Jbeil labels adapts it, without discarding what it learned.
This is why the two steps are worth doing in order: BDAPPV supplies breadth,
your 207 arrays supply local accuracy.

Upload `jbeil_seg.zip` (produced by `scripts/prepare_masks.py`) when prompted.
Skip this cell if you only want the general model.

In [ ]:
from google.colab import files
import zipfile, os

print("Upload jbeil_seg.zip ...")
up = files.upload()
name = list(up)[0]
with zipfile.ZipFile(name) as z:
    z.extractall("/content/local")
print("extracted:", os.listdir("/content/local"))

In [ ]:
# Fine-tune: start from the BDAPPV weights, drop the learning rate by 10x.
# A high LR here would overwrite the general knowledge we just paid 40 minutes
# to acquire -- the point is to nudge it toward Lebanon, not retrain it.
import torch
from torch.utils.data import DataLoader

FT_EPOCHS, FT_LR = 25, LR / 10

ft_tr = DataLoader(SegDS("/content/local/jbeil_seg", "train"), BATCH,
                   shuffle=True, num_workers=2, drop_last=True)
ft_va = DataLoader(SegDS("/content/local/jbeil_seg", "val"), BATCH, num_workers=2)
print(f"fine-tune on {len(ft_tr.dataset)} local crops, validate on {len(ft_va.dataset)}")

model.load_state_dict(torch.load("/content/solar_unet.pt")["state_dict"])
opt = torch.optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=FT_EPOCHS)

best_ft = -1.0
for ep in range(1, FT_EPOCHS + 1):
    model.train()
    for x, y in ft_tr:
        x, y = x.to(dev), y.to(dev)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(dev.type, enabled=dev.type == "cuda"):
            loss = crit(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    sched.step()

    model.eval(); inter = union = 0.
    with torch.no_grad():
        for x, y in ft_va:
            x, y = x.to(dev), y.to(dev)
            p = (torch.sigmoid(model(x)) > .5).float()
            inter += (p * y).sum().item()
            union += (p.sum() + y.sum() - (p * y).sum()).item()
    iou = inter / union if union else 1.
    print(f"ft epoch {ep:3d}  val_IoU={iou:.4f}")
    if iou > best_ft:
        best_ft = iou
        torch.save({"state_dict": model.state_dict(), "encoder": ENCODER,
                    "tile_size": TILE, "mean": MEAN, "std": STD,
                    "val_iou": iou, "epoch": ep}, "/content/solar_unet_lebanon.pt")

print(f"best fine-tuned val_IoU {best_ft:.4f}")
print("Compare against the BDAPPV-only score above: if fine-tuning did NOT help,")
print("use the general model -- 207 local examples can overfit.")

## 8 · Download the checkpoint

Save it to `models/solar_unet.pt` in your local repo, then restart the server —
the *Model checkpoint* dropdown will pick it up.

In [ ]:
from google.colab import files
# Download whichever scored better on validation.
files.download('/content/solar_unet.pt')            # BDAPPV general model
try:
    files.download('/content/solar_unet_lebanon.pt')  # fine-tuned, if you ran 7b
except Exception:
    pass
